<a href="https://colab.research.google.com/github/bkhn85dn/xac_dinh_va_phan_xu_diem_sai/blob/main/tap_chuan_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np
import json
import os
from collections import deque
from tabulate import tabulate
import pdb
epsilon = 1e-2

# Thứ tự phân cấp để đảm bảo lật từ "gốc" ra "ngọn"
SKELETON_HIERARCHY = {
    "left_shoulder": 0, "right_shoulder": 0, "left_hip": 0, "right_hip": 0,
    "left_elbow": 1, "right_elbow": 1, "left_knee": 1, "right_knee": 1,
    "left_hand": 2, "right_hand": 2, "left_ankle": 2, "right_ankle": 2
}

def get_orientation_flag(joints):
    rs, ls = np.array(joints["right_shoulder"]), np.array(joints["left_shoulder"])
    rh, lh = np.array(joints["right_hip"]), np.array(joints["left_hip"])
    mid_shoulders, mid_hips = (rs + ls) / 2, (rh + lh) / 2
    v_lr, v_spine = rs - ls, mid_shoulders - mid_hips
    forward_vec = np.cross(v_lr, v_spine)
    forward_vec /= np.linalg.norm(forward_vec)
    torso_center = (mid_shoulders + mid_hips) / 2

    flags = {}
    for name, pos in joints.items():
        dot_product = np.dot(np.array(pos) - torso_center, forward_vec)
        if abs(dot_product) < epsilon: flags[name] = 0
        else: flags[name] = 1 if dot_product > 0 else -1
    return flags

def calculate_clean_geo_error(data, names, stable_indices):
    coords1 = np.array([data["camera1"][names[i]] for i in stable_indices])
    coords2 = np.array([data["camera2"][names[i]] for i in stable_indices])
    def get_dist_mat(c):
        diff = c[:, np.newaxis, :] - c[np.newaxis, :, :]
        return np.sqrt(np.sum(diff**2, axis=-1))
    dist1, dist2 = get_dist_mat(coords1), get_dist_mat(coords2)
    errors = [np.linalg.norm(dist1[i] - dist2[i]) for i in range(len(stable_indices))]
    return errors

def get_pivot_mapping(lst):
    if not lst: return {}
    skeleton_edges = [
        ("left_shoulder", "left_elbow"), ("left_elbow", "left_hand"),
        ("right_shoulder", "right_elbow"), ("right_elbow", "right_hand"),
        ("left_hip", "left_knee"), ("left_knee", "left_ankle"),
        ("right_hip", "right_knee"), ("right_knee", "right_ankle"),
        ("left_shoulder", "right_shoulder"), ("left_hip", "right_hip"),
        ("left_shoulder", "left_hip"), ("right_shoulder", "right_hip")
    ]
    adj = {}
    all_joints = set()
    for u, v in skeleton_edges:
        adj.setdefault(u, []).append(v)
        adj.setdefault(v, []).append(u)
        all_joints.update([u, v])
    targets = set(lst)
    candidates = sorted(list(all_joints - targets))

    def get_reachable_targets(start_node):
        covered, queue, visited = set(), deque([start_node]), {start_node}
        while queue:
            curr = queue.popleft()
            for neighbor in adj.get(curr, []):
                if neighbor not in visited and neighbor in targets:
                    visited.add(neighbor); covered.add(neighbor); queue.append(neighbor)
        return covered

    candidate_coverage = {cand: get_reachable_targets(cand) for cand in candidates}
    priority_order = ["right_hip", "left_hip", "right_shoulder", "left_shoulder", "right_elbow", "left_elbow", "right_knee", "left_knee"]
    prio_map = {name: i for i, name in enumerate(priority_order)}

    uncovered, final_mapping = set(targets), {}
    while uncovered:
        best_cand, best_set = None, set()
        for cand in candidates:
            current_useful_cover = candidate_coverage[cand] & uncovered
            if not current_useful_cover: continue
            if best_cand is None or len(current_useful_cover) > len(best_set) or \
               (len(current_useful_cover) == len(best_set) and prio_map.get(cand, 999) < prio_map.get(best_cand, 999)):
                best_cand, best_set = cand, current_useful_cover
        if not best_cand: break
        final_mapping[tuple(sorted(list(best_set), key=lambda x: SKELETON_HIERARCHY.get(x, 0)))] = best_cand
        uncovered -= best_set
    return final_mapping

def solve_radial_bone_consistency(P_new, ray_A, target_dist, current_dist):
    """Giải phương trình bậc 2 để tìm A' trên tia ray_A sao cho |P_new - A'| = target_dist."""
    # |t*ray_A - P_new|^2 = D^2  => t^2 - 2t(ray_A.P) + |P|^2 - D^2 = 0
    b = -2 * np.dot(ray_A, P_new)
    c = np.sum(P_new**2) - target_dist**2
    delta = b**2 - 4*c
    if delta < 0: return None # Vô nghiệm hình học
    # Tính 2 giá trị t (khoảng cách từ O)
    t1 = (-b + np.sqrt(delta)) / 2
    t2 = (-b - np.sqrt(delta)) / 2

    # CHỌN NGHIỆM:
    # Chọn t nào khác với khoảng cách hiện tại (current_dist) nhất
    # để đảm bảo chúng ta đang thực hiện phép LẬT (Flip)
    t_final = t1 if abs(t1 - current_dist) > abs(t2 - current_dist) else t2

    # PHẢI NHÂN: Chuyển từ khoảng cách t sang tọa độ (x, y, z)
    return t_final * ray_A

def perform_advanced_correction(data, pivot_mapping):
    """
    Lật lan truyền cho cả 2 camera và kiểm tra bảo toàn độ dài xương.
    """
    cameras = ["camera1", "camera2"]
    original_coords_total = []
    flipped_coords_total = []
    full_bad_pose_names = []

    for cam_key in cameras:
        orig_cam = data[cam_key]
        working_cam = orig_cam.copy()

        for targets, pivot in pivot_mapping.items():
            for target_name in targets:
                full_bad_pose_names.append(f"{cam_key}_{target_name}")

                # 1. Tọa độ và độ dài xương ban đầu
                A_old = np.array(orig_cam[target_name])
                P_old = np.array(orig_cam[pivot])
                original_bone_length = np.linalg.norm(A_old - P_old)

                original_coords_total.append(A_old.tolist())

                # 2. Pivot mới (P_new) - có thể đã được cập nhật từ bước trước
                P_new = np.array(working_cam[pivot])
                dist_OA = np.linalg.norm(A_old)
                ray_A = A_old / dist_OA

                # 3. Tính toán tọa độ lật A_prime
                A_prime = solve_radial_bone_consistency(P_new, ray_A, original_bone_length, dist_OA)

                # Fallback nếu vô nghiệm hình học (tia nhìn không cắt mặt cầu xương)
                if A_prime is None:
                    dot_AP = np.dot(A_old, P_old)
                    k = (2 * dot_AP / np.sum(A_old**2)) - 1
                    A_prime = k * A_old

                # --- BƯỚC KIỂM TRA (ASSERTION) ---
                new_bone_length = np.linalg.norm(A_prime - P_new)
                diff = abs(new_bone_length - original_bone_length)

                assert diff < epsilon, \
                    f"LỖI BẢO TOÀN XƯƠNG tại {cam_key}: {target_name} -> {pivot}. " \
                    f"Sai lệch: {diff:.4f}m (Vượt ngưỡng {epsilon}m)"

                # 4. Lưu kết quả
                working_cam[target_name] = A_prime.tolist()
                flipped_coords_total.append(A_prime.tolist())

    return [original_coords_total, flipped_coords_total], full_bad_pose_names

def calculate_rigid_transform(data, stable_points):
    """
    Tìm s, R, T để biến đổi từ tập điểm A sang tập điểm B.
    Sử dụng thuật toán Umeyama (SVD).
    """
    # 1. Trích xuất tọa độ các điểm ổn định
    pts_cam1 = np.array([data["camera1"][p] for p in stable_points])
    pts_cam2 = np.array([data["camera2"][p] for p in stable_points])

    def get_transform(A, B):
        # A, B có kích thước (N, 3)
        n = A.shape[0]

        # Tính trọng tâm (Centroid)
        centroid_A = np.mean(A, axis=0)
        centroid_B = np.mean(B, axis=0)

        # Dời tập điểm về gốc tọa độ
        AA = A - centroid_A
        BB = B - centroid_B

        # Tính toán Scale (s)
        # s = sqrt( var(B) / var(A) )
        var_A = np.mean(np.sum(AA**2, axis=1))
        var_B = np.mean(np.sum(BB**2, axis=1))
        scale = np.sqrt(var_B / var_A)

        # Tính toán ma trận Xoay (R) bằng SVD
        # Ma trận hiệp phương sai H
        H = (AA.T @ BB) / n
        U, S, Vt = np.linalg.svd(H)

        R = Vt.T @ U.T

        # Xử lý trường hợp ma trận bị phản chiếu (Reflection)
        if np.linalg.det(R) < 0:
            Vt[2, :] *= -1
            R = Vt.T @ U.T

        # Tính toán Tịnh tiến (T)
        # T = centroid_B - s * R * centroid_A
        # Chuyển centroid_A về dạng cột (3, 1) để nhân ma trận
        T = centroid_B.reshape(3, 1) - scale * (R @ centroid_A.reshape(3, 1))

        # Trả về các mảng numpy để dễ tính toán sau này
        return scale, R, T

    # Tính toán cho 2 chiều
    s1, R1, T1 = get_transform(pts_cam1, pts_cam2)
    s2, R2, T2 = get_transform(pts_cam2, pts_cam1)

    # Trả về 1 tuple duy nhất có 6 phần tử
    return s1, R1, T1, s2, R2, T2

def final_analysis(data, reduced):
    # (Giữ nguyên logic phân tích của bạn, trả về error_pose_names)
    names = list(data["camera1"].keys())
    flags1, flags2 = get_orientation_flag(data["camera1"]), get_orientation_flag(data["camera2"])
    conflict_indices, stable_indices = [], []
    for i, name in enumerate(names):
        f1, f2 = flags1[name], flags2[name]
        if (f1 == 1 and f2 == -1) or (f1 == -1 and f2 == 1): conflict_indices.append(i)
        else: stable_indices.append(i)

    clean_errors = calculate_clean_geo_error(data, names, stable_indices)
    hybrid_threshold = (np.mean(clean_errors) + np.median(clean_errors)) / 2 if clean_errors else 0

    final_results = []
    for i in conflict_indices: final_results.append({"name": names[i], "error": 999.0, "status": "XUNG ĐỘT"})
    for i, idx in enumerate(stable_indices):
        err = clean_errors[i]
        final_results.append({"name": names[idx], "error": err, "status": "LỚN" if err >= hybrid_threshold else "Ổn định"})

    final_results.sort(key=lambda x: x['error'], reverse=True)
    result_1 = [item["name"] for item in final_results if item["status"] != "Ổn định"]
    result_2 = [item["name"] for item in final_results if item["status"] == "Ổn định"]
    return [result_1, result_2]

# --- MAIN PROCESS ---
if __name__ == "__main__":
    file_path = "pose_data.json" # Giả sử tên file
    if os.path.exists(file_path):
        with open(file_path, 'r') as f:
            data = json.load(f)

            # 1. Tìm các điểm lỗi
            bad_poses, good_poses = final_analysis(data, reduced=True)
            print(f"Bad Poses: {bad_poses}")
            print(f"Good Poses: {good_poses}")
            s1, R1, T1, s2, R2, T2 = calculate_rigid_transform(data, good_poses)
            #pdb.set_trace()
            # 2. Ánh xạ Pivot
            pivot_map = get_pivot_mapping(bad_poses)

            # 3. Thực hiện lật lan truyền và lấy cấu trúc 3D
            result_3d, pose_order = perform_advanced_correction(data, pivot_map)

            print("\n--- KẾT QUẢ CẤU TRÚC 3 CHIỀU ---")
            #print(f"Thứ tự các Pose trong danh sách: {pose_order}")
            #print(f"Hàng 1 (Gốc): {result_3d[0]}")
            #print(f"Hàng 2 (Lật): {result_3d[1]}")
            # Chuẩn bị dữ liệu để in
            num_poses_per_cam = len(pose_order) // 2
            table_data = []

            for i in range(num_poses_per_cam):
                # Lấy tên khớp (bỏ tiền tố 'camera1_')
                pose_name = pose_order[i].replace("camera1_", "")

                # Chỉ số của Cam 1 (i) và Cam 2 (i + offset)
                idx1 = i
                idx2 = i + num_poses_per_cam

                # Tọa độ Cam 1
                c1_goc = [round(v, 4) for v in result_3d[0][idx1]]
                c1_lat = [round(v, 4) for v in result_3d[1][idx1]]

                # Tọa độ Cam 2
                c2_goc = [round(v, 4) for v in result_3d[0][idx2]]
                c2_lat = [round(v, 4) for v in result_3d[1][idx2]]

                # Gộp vào một hàng
                table_data.append([
                    pose_name,
                    f"({c1_goc[0]}, {c1_goc[1]}, {c1_goc[2]})",
                    f"({c1_lat[0]}, {c1_lat[1]}, {c1_lat[2]})",
                    f"({c2_goc[0]}, {c2_goc[1]}, {c2_goc[2]})",
                    f"({c2_lat[0]}, {c2_lat[1]}, {c2_lat[2]})"
                ])

            # Định nghĩa lại Headers cho bảng 5 cột
            headers = [
                "Khớp (Pose)",
                "Cam 1 (Gốc)", "Cam 1 (Lật)",
                "Cam 2 (Gốc)", "Cam 2 (Lật)"
            ]

            print("\nSO SÁNH DỮ LIỆU POSE 3D HỢP NHẤT (GỐC vs LẬT):")
            print(tabulate(table_data, headers=headers, tablefmt="grid"))

Bad Poses: ['left_knee', 'left_ankle', 'right_ankle', 'left_hand', 'left_elbow', 'right_hand']
Good Poses: ['left_hip', 'left_shoulder', 'right_knee', 'neck', 'right_shoulder', 'right_hip', 'right_elbow']
> /tmp/ipykernel_2175/3257590606.py(196)<cell line: 0>()
    194             pdb.set_trace()
    195             # 2. Ánh xạ Pivot
--> 196             pivot_map = get_pivot_mapping(bad_poses)
    197 
    198             # 3. Thực hiện lật lan truyền và lấy cấu trúc 3D

--KeyboardInterrupt--

KeyboardInterrupt: Interrupted by user

--- KẾT QUẢ CẤU TRÚC 3 CHIỀU ---

SO SÁNH DỮ LIỆU POSE 3D HỢP NHẤT (GỐC vs LẬT):
+---------------+----------------------------+----------------------------+---------------------------+----------------------------+
| Khớp (Pose)   | Cam 1 (Gốc)                | Cam 1 (Lật)                | Cam 2 (Gốc)               | Cam 2 (Lật)                |
+===============+============================+============================+===========================+===========